## Ideal scenario
Simulation BB84 code
- No noise 
- No eavesdropper
- Perfect transmission (communication)

#### Expected
- QBER=0
- Key agreement=100%
- stable key generation rate


In [2]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit_aer.primitives import SamplerV2

In [3]:
sampler=SamplerV2()

In [4]:
def generate_bits(n):
    return np.random.randint(2, size=n)

In [5]:
#0=>Z, 1=>X
def generate_bases(n):
    return np.random.randint(2, 
                             size=n)

In [7]:
def encode_qubits(bits, bases):
    circuits=[]
    for bit,basis in zip(bits,bases):
        qc=QuantumCircuit(1,1)
        if bit==1:
            qc.x(0)
        if basis==1: 
        #apply hadamards gate
            qc.h(0)
        circuits.append(qc)
    return circuits


In [16]:
def measure_qubits(circuits,bases):
    results=[]
    for qc,basis in zip(circuits,bases):
        mqc=qc.copy()
        if basis==1:
           mqc.h(0)
        mqc.measure(0,0)

        job=sampler.run([mqc], shots=1)
        result=job.result()
        bitarray=result[0].data.c
        counts=bitarray.get_int_counts()
        measured_bit=max(counts, key=counts.get)
        results.append(measured_bit)
    return np.array(results)

In [9]:
def sift_keys(alice_bits, alice_bases, bob_res, bob_bases):
    mask=alice_bases==bob_bases
    return alice_bits[mask], bob_res[mask]

In [19]:
def qber_calc(alice_key, bob_key):
    if len(alice_key)==0:
        return 0
    return np.sum(alice_key!=bob_key)/len(alice_key)

In [11]:
def key_agreement_rate(alice_key, bob_key):
    return np.sum(alice_key==bob_key)/len(alice_key)

In [12]:
def key_generation_rate(sifted_len, total_qubs):
    return sifted_len/total_qubs

In [20]:
def run_bb84(n=1000):
    alice_bits=generate_bits(n)
    alice_bases=generate_bases(n)
    bob_bases=generate_bases(n)

    circuits=encode_qubits(alice_bits, alice_bases)
    bob_results=measure_qubits(circuits, bob_bases)

    alice_key, bob_key=sift_keys(
        alice_bits, alice_bases, bob_results, bob_bases
    )
    qber=qber_calc(alice_key, bob_key)
    agreement=key_agreement_rate(alice_key, bob_key)
    kgr=key_generation_rate(len(alice_key), n)

    return {
        "QBER": qber,
        "agreement": agreement,
        "keyRate": kgr,
        "keyLength": len(alice_key)
    }

In [21]:
def run_experiments(num_runs=100, n=1000):
    qbers = []
    agreements = []
    key_rates = []
    
    for _ in range(num_runs):
        result = run_bb84(n)
        qbers.append(result["QBER"])
        agreements.append(result["agreement"])
        key_rates.append(result["keyRate"])
    
    return {
        "QBER_mean": np.mean(qbers),
        "QBER_std": np.std(qbers),
        "agreement_mean": np.mean(agreements),
        "keyRate_mean": np.mean(key_rates)
    }

In [22]:
results = run_experiments(num_runs=100, n=1000)

print("=== Ideal Channel Results ===")
for k, v in results.items():
    print(f"{k}: {v:.4f}")

=== Ideal Channel Results ===
QBER_mean: 0.0000
QBER_std: 0.0000
agreement_mean: 1.0000
keyRate_mean: 0.5031
